# 03 Character-level CNN (Model 2)

The CNN of Gregório et al. (2024), Figure 1: a character embedding, six convolutions with max-pooling, and a dense classifier. It is implemented in `src/models/charcnn.py`, trained with `src/models/train.py`, and scored with the same metrics as the baselines (`src/models/metrics.py`).

It is run in two setups, like Model 1: **binary** (target DGA/legitimate) and **family-first** (target `legit` or a DGA family, collapsed with P(DGA) = 1 - P(legit)). The input is the full domain name as it is, TLD included (nothing is split off), and both setups use the sample weights. Metrics are computed on the DGA/legitimate decision, on the validation set; the test set is kept for the final results.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

## Settings

Training follows the paper (Table 2): Adam, learning rate 1e-4, batch size 250, L1 1e-5 and L2 1e-4, 20 epochs, keeping the best epoch on validation (here by macro-F1). On a CPU one epoch over the full training set takes more than an hour, so use a GPU (e.g. Colab) for `SAMPLE = None`.

In [ ]:
SAMPLE = 100_000   # None = full training set
EPOCHS = 20
SEED = 0
EVAL_SET = "val"   # "test" only for the final results
WEIGHTED = True    # use the sample weights in the loss
CHECKPOINTS = ROOT / "checkpoints"   # trained models are saved here (not committed)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from src.data.dataset import load_splits
from src.models.charcnn import CharCNN
from src.models.encoding import class_codes, encode
from src.models.metrics import binary_metrics, recall_per_family
from src.models.train import fit, p_dga, pick_device, predict_proba, set_seed

device = pick_device()
print("device:", device, torch.cuda.get_device_name() if device.type == "cuda" else "")

train_full, val, test = load_splits()
train = train_full.sample(n=SAMPLE, random_state=SEED) if SAMPLE else train_full
evaluate = test if EVAL_SET == "test" else val
y_val = (val["label"] == "dga").astype(int).to_numpy()
y_eval = (evaluate["label"] == "dga").astype(int).to_numpy()
print(f"train: {len(train):,} rows (of {len(train_full):,}), "
      f"{EVAL_SET}: {len(evaluate):,} rows")

## Input encoding

The full domain name, TLD included, is the input. Each character becomes its ASCII code, padded with 0 at the end to 93 characters, the longest domain in the dataset, so no domain is cut (the paper used 70, the longest in its data). Classes are numbered from the full training set, so every family has an output even if a sample misses it.

In [ ]:
x_train, x_val, x_eval = (encode(part["domain"]) for part in (train, val, evaluate))
classes = {"label": sorted(train_full["label"].unique()),
           "family": sorted(train_full["family"].unique())}
print(f"input: {x_train.shape}, {len(classes['family'])} family classes")
print(train["domain"].iloc[0], "->", x_train[0][:20], "...")

## The model

Figure 1 of the paper with a 93-character input: the flattened size is 93 x 30 = 2790 (70 x 30 = 2100 in the paper).

In [ ]:
model = CharCNN({"family": len(classes["family"])})
print(model)
print(f"{sum(p.numel() for p in model.parameters()):,} parameters")

## Training

One model per setup. The model is kept at its best validation epoch and saved to `checkpoints/`.

In [ ]:
SETUPS = {"binary": "label", "family": "family"}
CHECKPOINTS.mkdir(exist_ok=True)

results, histories, p_dga_eval = {}, {}, {}
for setup, task in SETUPS.items():
    print(f"--- {setup} ({len(classes[task])} classes)")
    set_seed(SEED)
    model = CharCNN({task: len(classes[task])})
    histories[setup] = fit(model, x_train, {task: class_codes(train[task], classes[task])},
                           train["weight"] if WEIGHTED else None, x_val, y_val, classes,
                           decide=task, epochs=EPOCHS, seed=SEED, device=device)
    torch.save(model.state_dict(), CHECKPOINTS / f"charcnn_{setup}_seed{SEED}.pt")
    p_dga_eval[setup] = p_dga(predict_proba(model, x_eval), task, classes)
    results["Model 2", setup] = binary_metrics(y_eval, p_dga_eval[setup])

## Results (DGA/legitimate decision)

In [ ]:
table = pd.DataFrame(results).T.round(4)
table.index.names = ["model", "setup"]
table

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.4))
for setup, history in histories.items():
    ax1.plot(history.index, history["train_loss"], marker="o", label=setup)
    ax2.plot(history.index, history["macro_f1"], marker="o", label=setup)
ax1.set_xlabel("epoch")
ax1.set_ylabel("training loss")
ax1.set_title("Training loss")
ax2.set_xlabel("epoch")
ax2.set_ylabel("macro-F1")
ax2.set_title("Validation macro-F1")
for ax in (ax1, ax2):
    ax.legend()
fig.tight_layout()
plt.show()

### Recall per DGA family

Recall of each DGA family in the two setups, families ordered from largest to smallest.

In [ ]:
family_sizes = train_full.loc[train_full["label"] == "dga", "family"].value_counts()
recall = pd.DataFrame({
    setup: recall_per_family(evaluate["family"], y_eval, (p >= 0.5).astype(int))
    for setup, p in p_dga_eval.items()}).reindex(family_sizes.index)
recall.columns = ["binary", "family-first"]

fig, ax = plt.subplots(figsize=(12, 3.6))
x = np.arange(len(recall))
for setup, marker in [("binary", "o"), ("family-first", "D")]:
    ax.scatter(x, recall[setup], s=24, marker=marker, label=setup, zorder=3)
ax.set_xlim(-1, len(recall))
ax.set_ylim(-0.03, 1.05)
ax.set_xlabel(f"DGA family, largest to smallest ({len(recall)} families)")
ax.set_ylabel("recall")
ax.set_title("Recall per DGA family, binary vs. family-first")
ax.legend(loc="lower left")
fig.tight_layout()
plt.show()

## Families the CNN misses most

The 10 DGA families with the lowest recall in the family-first setup.

In [ ]:
recall.sort_values("family-first").head(10).round(3)